# Naive Bayes i Classificació

### Instruccions

+ L'exàmen consta de tres preguntes. 
+ Per a poder contestar les preguntes, heu de descarregar el fitxer `cyberbullying_tweets.csv` del campus virtual.
+ Podeu usar també les funcions del notebook de la vostra pràctica.


## 1. Lectura de les dades

In [16]:
import pandas as pd
import numpy as np
import re
import sklearn
import nltk
from nltk.corpus import stopwords
import matplotlib.pyplot as plt
%matplotlib inline

In [17]:
df = pd.read_csv('./data/cyberbullying_tweets.csv')
df

,tweet_text,cyberbullying_type
0,"In other words #katandandre, your food was cra...",not_cyberbullying
1,Why is #aussietv so white? #MKR #theblock #ImA...,not_cyberbullying
2,@XochitlSuckkks a classy whore? Or more red ve...,not_cyberbullying
3,"@Jason_Gio meh. :P thanks for the heads up, b...",not_cyberbullying
4,@RudhoeEnglish This is an ISIS account pretend...,not_cyberbullying
...,...,...
47687,"Black ppl aren't expected to do anything, depe...",ethnicity
47688,Turner did not withhold his disappointment. Tu...,ethnicity
47689,I swear to God. This dumb nigger bitch. I have...,ethnicity
47690,Yea fuck you RT @therealexel: IF YOURE A NIGGE...,ethnicity


In [18]:
from sklearn.model_selection import train_test_split

df_tweets_train, df_tweets_test = train_test_split(df, test_size=0.2)

In [19]:
# No modificar aquesta cel·la, s'encarrega de fer el procés més eficient.
# Intenteu entendre quà fa aquesta cel·la
def memo(f):
    class memodict(dict):
        def __init__(self, f):
            self.f = f
        def __call__(self, *args):
            return self[args]
        def __missing__(self, key):
            ret = self[key] = self.f(*key)
            return ret
    return memodict(f)

@memo    
def standardize(word):
    """
    :param word: paraula a estandaritzar
    :return : paraula estandaritzada
    """
    # Convertim a minúscules
    word = word.lower()
    
    # Eliminem els simbols no permesos
    word = re.sub(r"[^a-z0-9@#_ ]", "", word)

    # Eliminem espais innecessaris
    word = re.sub(r"\s+", " ", word).strip()
    
    # Remove URLs
    word = re.sub(r'^https?:\/\/.[\r\n]', '', word)
    
    # Remove punctuation and others
    word = re.sub(r'[^a-z\s]', '', word)
    
    # Remove repeated characters
    re.sub(r'(.)\1{3,}', r'\1', word)
    # Remove extra spaces
    word = re.sub(r'\s+', ' ', word).strip()
    
    return word

# Creem aquesta funció auxiliar per cridar standardize
def standarize_text(word):
    """
    :param word: paraula a estandaritzar
    :return : paraula estandaritzada utilitzant standardize
    """
    return standardize(word)

def count_words(df):
    """
    :param df: DataFrame amb les piulades i la informació associada
    :return : Diccionari amb el format {word : {n_ocur: valor, n_tweets: valor}, ...}
    """
    # Utilizem apply amb la funció standarize_text() que acabem de definir
    standarizedText =  df["tweet_text"].apply(standarize_text)
    
    # IDEA: Crear un df amb totes les paraules i fer un value_counts()
    # Després tornarem al df original y comptarem els tweets on surt cada paraula
    
    # Combinem tots els tweets en un únic text i dividim en paraules
    all_words = " ".join(standarizedText).split(" ")

    # Comptar totes les ocurrències de paraules (n_ocur)
    word_count = pd.Series(all_words).value_counts()

    # Comptar en quants tweets apareix cada paraula (n_piu)
    # Convertim cada tweet en un conjunt de paraules úniques
    unique_words_per_tweet = standarizedText.apply(lambda x: set(x.split()))
    tweet_count = pd.Series([word for tweet in unique_words_per_tweet for word in tweet]).value_counts()
    
    # Construim el diccionari directament
    dicc = {
        word: {"n_ocur": word_count[word], "n_piu": tweet_count[word]}
        for word in word_count.index if word.strip()
    }

    return dicc

def count_words_categories(df):
    """
    Funció que ha de constuir un diccionari que conté la freqüència de les 
    paraules i el número de piulades on ha aparegut. 
    Aquesta informació ha de ser dividida per diferents categories de cyberbullying.
    
    :param df: DataFrame amb les piulades i la informació associada
    :return : Diccionari amb el format {label : {word : {n_ocur: valor, n_news: valor} } }
    """
    words_topic = {}

    def eachTopic(group):
        # Count words on this topic and save to dictionary
        words_topic[group['cyberbullying_type'].iloc[0]] = count_words(group)
        
    # IDEA: Agafar tots els tipus de cyberbullying
    # i cridar la funció eachTopic per cadascun d'ells
    categories = df["cyberbullying_type"].unique()

    for category in categories:
        data = df[df["cyberbullying_type"] == category]
        eachTopic(data)
    
    return words_topic

def topNwords(df, words, N, skip=[]):
    """
    :param df: DataFrame amb les piulades i la informació associada
    :param words: diccionari amb les paraules i la seva frequencia
    :param N: número de paraules més representatives que volem considerar
    :return : Diccionari amb el format {categoria1: llista_top_words_cat_1,  
                                        categoria2: llista_top_words_cat_2, ...} 
    """
    top_words=dict()
    
    # IDEA: Iterar pel diccionari amb les paraules
    # i les ocurrencies i buscar les N més representatives
    
    # Guardem les categories de cyberbullying
    categories = df["cyberbullying_type"].unique()

    for category in categories:
        # Guardem la informació de cada categoria
        category_words = words[category]
                
        # Per totes les paraules de la categoria mirem si és vàlida
        filtered_words = {word: freq for word, freq in category_words.items() if word not in skip}

        # Ordenem les paraules per n_ocur de forma descendent
        sorted_words = sorted(filtered_words.items(), key=lambda x: x[1]["n_ocur"], reverse=True)

        # Seleccionem les N paraules més representatives
        top_category_words = [word for word, _ in sorted_words[:N]]

        # Afegim la llista al diccionari
        top_words[category] = top_category_words
    
    return top_words

def create_features(df, top_words): 
    """
    :params df: DataFrame amb les piulades i la informació associada
    :params top_words: ha de ser el diccionari que retorna topNWords
    :return : diccionari o pd.Series que conté un np.array per a 
        cadascuna de les piulades amb el vector de característiques corresponent.
    """

    # IDEA: Guardar totes les paraules úniques escrites i per cada
    # comentari veure si aquesta paraula surt o no.
    # Per fer això utilitzarem moltes llistes, 
    # set i list/set-comprehension. És fundamental (i de fet vam tenir problemes)
    # que es controlin bé els indexos perque es fan moltes iteracions
    # i és fàcil liar-la
    
    # Combinar totes les paraules top de totes les categories
    unique_words = list(set(word for words in top_words.values() for word in words))

    # Crear un índex per accedir ràpidament a les paraules
    word_index = {word: idx for idx, word in enumerate(unique_words)}
    
    # Inicialitzar el diccionari de vectors de característiques
    dict_feat_vector = {}

    # Utilizem apply amb standarize_text
    standarizedText =  df["tweet_text"].apply(standarize_text)

    # Agafem l'index i el tweet
    # És important agafar aquest index, sinó perdem 
    # consistencia amb la resta de la taula
    for idx, tweet in standarizedText.items():
        # Dividim el tweet en paraules
        tweet_words = set(tweet.split())
        
        # Inicialitzem el vector de característiques
        feature_vector = np.zeros(len(unique_words), dtype=int)
        
        # Marquem les paraules que hi ha al tweet
        for word in tweet_words:
            if word in word_index:
                feature_vector[word_index[word]] = 1
        
        # Assignem el vector
        dict_feat_vector[idx] = feature_vector
    
    return dict_feat_vector

def naive_bayes_learn(df, feats):
    """
    :params df: DataFrame amb les piulades i la informació associada
    :params feats: vector de característiques de cada piulada
    :return : probabilitats marginals condicionades
    """
    # IDEA: Implementar la formula detallada adalt i trobar
    # A, B i M. Això ho farem jugant amb les categories
    # i amb el vector de característiques
    
    # Guardem les categories i la quantitat que hi ha
    categories = df["cyberbullying_type"].unique()

    # M és el total de categories
    M = len(categories)

    # Inicialitzem la estructura per guardar les probabilitats de cada categoria
    probs = {category: [] for category in categories}

    # Iterem per totes les categories per calcular les probabilitats
    for category in categories:
        # Agafem les piulades d'aquesta categoria i les guardem com una matriu
        category_indices = df[df["cyberbullying_type"] == category].index
        category_feats = np.array([feats[i] for i in category_indices])
        
        # A és total de piulades on surt una paraula en particular
        # Podem fer la suma per columnes perque cada fila és una piulada 
        # amb 0 o 1 a la posició i-ésima en cas de tener la paraula o no
        # per tant, fer la suma per columnes equival a comptar quantes piulades
        # tenen una paraula en particular
        A = np.sum(category_feats, axis=0)

        # B és el total de piulades hi ha a aquesta categoria
        B = len(category_feats)

        # Apliquem la correcció de Laplace
        probs[category] = (A + 1) / (B + M)

    return probs
    
import sys
from IPython import embed
def naive_bayes(df_train, feat_train, feat_test=None, df_test=None):
    """
    Funció que implementa el clasificador Naive_Bayes.
    
    Si df_test no és None, ha de calcular l'encert sobre les dades de test. És a dir,
    després de classificar feat_test ha de comparar la classificació amb la classe
    real i dir (print) quin percentatge d'encert ha obtingut.
    
    :param df_train: DataFrame amb les piulades que s'utilitzaran per l'entrenament
    :param feat_train: Diccionari amb els vectors de caracteristiques de cada tweet de l'entrenament
    :param feat_test: Diccionari amb els vectors de caracteristiques de cada tweet de test
    :param df_test: DataFrame amb les piulades que s'utilitzaran pel test
    
    :return : Una serie on l'index correspon amb els indexos de df_test i els valors són la
              classificació retornada per Naive Bayes
    """
    probs = naive_bayes_learn(df_train, feat_train)
    p_of_cat = count_words_categories(df_train)
    p_total = len(p_of_cat.keys())
    
    def eachFeats(row):
        id, feat = row
        p_max = float('-inf')
        p_cat = 0

        # IDEA: Agrupar tota la informació i aplicar les formules
        # observem a la demostració de l'us dels logaritmes
        # que s'ha detallat a la part superior        
        
        for category in probs:
            # Speed up by using numpy
            # inv is the inverse of features, 0 where 1 and 1 where 0
            inv_feat = 1 - feat
            
            # Probs * feats is the probability of being there, while
            # inv - inv * feat = 1 - (0, 1, 0... inverses) * probs, probability of not being there
            prob_there = probs[category]
            prob_not_there = 1 - probs[category]
            
            # Sum of logs [vs] underflow caused by mul of probs
            log_sum = np.sum(feat * np.log(prob_there) + inv_feat * np.log(prob_not_there))

            # Take the max, do it now to avoid extra-loops
            if log_sum > p_max:
                p_max = log_sum
                p_cat = category
        return id, p_cat
    
    data = map(eachFeats, feat_test.items())
    data = pd.Series(dict(data))
    correct = data == df_test['cyberbullying_type']
    print("Accuracy: {}".format(correct.sum() / correct.size))
    
    return correct.sum() / correct.size

def acc(train, test, N=10, skip=[]):
    """
    Retorna l'accuracy donades certes condicions
    
    :param train: DataFrame per entrenar el model
    :param test: DataFrame per testejar el model
    :param N: número de paraules més representatives que volem considerar 
    :param skip: llista de stopwords
    :return : Escalar (float) corresponent a l'accuracy
    """  
    
    words_topics = count_words_categories(train)
    top_words = topNwords(train, words_topics, N, skip)

    feat_train = create_features(train, top_words)
    feat_test = create_features(test, top_words)
    # probs = naive_bayes_learn(df_tweets_train, feat_train)
    accuracy = naive_bayes(train, feat_train, feat_test, test)
    return accuracy

N = 40
skip_top = []
accuracy = acc(df_tweets_train, df_tweets_test, N=N, skip=skip_top)

Accuracy: 0.7315232204633609


## Exercici 1 (3 punts)

+ Completa la següent funció, la qual rep una piulada per paràmetre i retorna la seva categoria predita.
+ Comprova que funcioni escrivint alguna piulada d'exemple. 

In [29]:
def predict_category(tweet, df_train, top_words, probs):
    """
    :param tweet: The tweet text to classify
    :param df_train: DataFrame with the training tweets and associated information
    :param top_words: Dictionary of top words for each category
    :param probs: Dictionary of conditional probabilities for each category
    :return: Predicted category of the tweet
    """
    
    tweet_words = [standardize(word) for word in tweet.split()]

    #Completa:
    most_repeated_words = set([word for words in top_words.values() for word in words])
    word_index = {word: idx for idx, word in enumerate(most_repeated_words)}
    
    #Completa:
    feature_vector = np.zeros(len(most_repeated_words), dtype=int)
    for word in tweet_words:
            if word in word_index:
                feature_vector[word_index[word]] = 1

    p_max = float('-inf')
    p_cat = None
        

    for category in probs:
        
        inv = 1 - feature_vector
        here = probs[category] * feature_vector
        not_here = inv - inv * probs[category]
        conditional_prob = here + not_here
        sum_log = sum(np.log(conditional_prob))

        if sum_log > p_max:
            p_max = sum_log
            p_cat = category

    return p_cat

In [30]:
words_topics = count_words_categories(df_tweets_train)
top_words = topNwords(df_tweets_train, words_topics, N)

feats_train = create_features(df_tweets_train, top_words)
feats_test = create_features(df_tweets_test, top_words)
probs = naive_bayes_learn(df_tweets_train, feats_train)
    
tweet = "I hate man"
predicted_category = predict_category(tweet, df_tweets_train, top_words, probs)
print(predicted_category)

other_cyberbullying


## Exercici 2 (5 punts)

+ Crea la funció `fix_tweet` que rebi una piulada, la seva categoria predita segons el vostre criteri i comprovi si el model la prediu correctament fent servir la funció anterior. En cas contrari, afegirà paraules aleatoriament al tweet per fins que el predigui correctament. 
+ Pensa quines paraules hauria d'afegir i comprova que l'estratègia funciona.

In [34]:
def fix_tweet(tweet, correct_category, df_train, top_words, probs):
    """
    :param tweet: The tweet text to classify
    :param correct_category: The correct category of the tweet
    :param df_train: DataFrame with the training tweets and associated information
    :param top_words: Dictionary of top words for each category
    :param probs: Dictionary of conditional probabilities for each category
    :return: Modified tweet and whether the category is fixed or not
    """
    
    modified_tweet = tweet
    predicted_category = None
    add_words = top_words[correct_category]
    i = 0
    
    while predicted_category != correct_category and i < len(add_words):
        modified_tweet += " " + add_words[i]
        predicted_category = predict_category(modified_tweet, df_train, top_words, probs)
        i += 1
    
    return modified_tweet, predicted_category

In [36]:
# Test: per el tweet anterior, "I hate man" hauria d'afegir paraules per tal d'obtenir la cateogria correcte: gender

tweet = "I hate man"
correct_category = "gender"
modified_tweet, cat = fix_tweet(tweet, correct_category, df_tweets_train, top_words, probs)
print(modified_tweet)
print("New Category: ",cat)

I hate man a the rape
New Category:  gender


## Exercici 3 (2 punts)

Explica de forma clara i concisa en què consisteix fer un model "Naive Bayes" d'un tweet.

 
> Un model **Naive Bayes** per classificar tweets funciona calculant quina categoria és més probable segons les paraules que conté el tweet.
> 1. Primer, aprèn les probabilitats de cada paraula en cada categoria a partir dels tweets d’entrenament.
> 2. Després, quan arriba un nou tweet, combina aquestes probabilitats per calcular la categoria més probable.
> 3. Finalment, assigna al tweet la categoria amb la puntuació més alta.
> És ràpid i senzill perquè assumeix que les paraules són independents, encara que no ho siguin del tot.



### Vectors de Característiques: Teoria i Preguntes

Són representacions matemàtiques d'un punt de dades en un espai multi-dimensional, on cada dimensió correspon a un atribut o característica específica de les dades. En el context de Naive Bayes, aquests vectors representen les dades d'entrada que el model utilitza per calcular les probabilitats a partir de les característiques observades.

#### Per a què serveixen en el context de Naive Bayes?
1. **Representació de dades**: Cada vector representa una instància del conjunt de dades amb els seus atributs. Per exemple, en text classificació, les dimensions podrien ser la presència o freqüència de paraules.
2. **Entrada al model**: Naive Bayes assumeix que les característiques són independents i les utilitza per calcular la probabilitat condicional de cada classe.
3. **Càlcul eficient**: La representació en vectors permet realitzar càlculs probabilístics fàcils gràcies a la separació de les característiques.



### Quina diferència hi ha entre `count_words_categories` i `create_features`?

**`count_words_categories`:**  
- Compta la freqüència de cada paraula en cada categoria, ajudant a identificar quines paraules són més representatives.  
- Retorna un diccionari amb paraules i les seves freqüències agrupades per categoria.

**`create_features`:**  
- Converteix un conjunt de tweets en vectors de característiques binàries.  
- Indica si una paraula apareix o no al tweet, basant-se en les paraules representatives prèviament seleccionades.

---

### Què fa `naive_bayes_learn` i per què és important?

**`naive_bayes_learn`** calcula les probabilitats condicionals que utilitza el model per fer prediccions.  

- **Entrades:** Tweets del conjunt d'entrenament i els seus vectors de característiques.  
- **Sortida:** Un diccionari amb la probabilitat de cada paraula donada una categoria.  
- És essencial perquè Naive Bayes es basa en aquestes probabilitats per determinar la categoria més probable d'un nou tweet.

---

### Com milloraries aquest sistema?

Algunes millores possibles:

- **Processament del text:**  
  - Afegir *stemming* (reducció de paraules al seu arrel comú) o eliminació de *stopwords* per optimitzar les dades.

- **Pesos:**  
  - Considerar la freqüència de les paraules en lloc de vectors binaris per a un model més refinat (TF-IDF).

- **Validació:**  
  - Introduir una fase de *cross-validation* per avaluar millor el rendiment del model.

- **Altres models:**  
  - Provar algoritmes més avançats (per exemple, models basats en *embeddings* com Word2Vec o *transformers*).
